# Digital heritage in video games: a quick start

This notebook walks through the `gameheritage` catalogue: video games' versions of real places,
side by side with the places themselves.

Install the package first:

```bash
pip install "git+https://github.com/cantayus/gameheritage@v0.1.0"
```

In [ ]:
import gameheritage as gh

cat = gh.load()
cat

## 1. Search for a place or a game

In [ ]:
cat.search("Notre-Dame")[["game_title", "in_game_name", "site_name", "century", "era"]]

In [ ]:
cat.search("Kyoto")[["game_title", "in_game_name", "site_name", "era"]]

## 2. All the real places in one game

`matches()` gives one row per in-game version of a real place: the game, the place, its date and era,
how the match was made, and one game image and one real photograph with their URLs and credits.

In [ ]:
unity = cat.matches(game="Assassin's Creed Unity")
unity[["in_game_name", "site_name", "year", "century", "era", "n_game_images", "n_real_images"]].head(10)

## 3. Through the ages

In [ ]:
cat.timeline()

In [ ]:
medieval_france = cat.matches(era="Middle Ages", country="France")
medieval_france[["game_title", "site_name", "century", "locality"]]

In [ ]:
# Which games show the most ancient places?
ancient = cat.matches(era="Ancient world")
ancient.groupby("game_title")["site_id"].nunique().sort_values(ascending=False).head(10)

## 4. Where in the world

In [ ]:
places = cat.places()
places["country"].value_counts().head(10)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

xy = places.assign(lat=pd.to_numeric(places["latitude"], errors="coerce"),
                   lon=pd.to_numeric(places["longitude"], errors="coerce")).dropna(subset=["lat", "lon"])
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(xy["lon"], xy["lat"], s=6, alpha=0.6)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title(f"{len(xy):,} real places shown in {cat.summary()['games']} games")
plt.show()

## 5. Look at a pair

`gh.show()` puts the game image and the real photograph side by side, with the full credit line of each.
It needs an internet connection: the images are fetched from the project's website.

In [ ]:
row = cat.matches(site="Notre-Dame de Paris").iloc[0]
gh.show(row)

## 6. Download images with their credits

`gh.download()` saves the images behind any table and writes `ATTRIBUTION.csv` next to them.
`size="web"` (default) gets the project's 1200 px copies; `size="original"` gets the full files from
Wikimedia Commons and Fandom. Game images are copyrighted by their publishers; check your use.

In [ ]:
result = gh.download(unity.head(5), "unity_images/")
result[["image_id", "role", "copy", "status", "license"]]

In [ ]:
# Only photographs without share-alike terms, at full size
free = gh.download(unity.head(5), "unity_originals/", role="real", size="original",
                   licenses=["Public domain", "CC0", "CC BY"])
free[["image_id", "copy", "status", "license"]]

## 7. What the columns mean, and how to cite

In [ ]:
gh.describe("depictions")

In [ ]:
print(gh.cite())

**Note.** In release 0.1.0 every match was made automatically (`verified == "false"`).
Check the matches you rely on: the in-game name, the images and `basis_note` show how each was made.